## Подключение Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Проверка GPU

In [2]:
!nvidia-smi

Sun May 10 16:29:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## Установка библиотек

In [5]:
!pip install ultralytics opencv-python matplotlib pandas pyyaml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s eta 0:00:00


## Проверка папок

In [6]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/smart-parking-vision")
DATA_DIR = BASE_DIR / "data"

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

print("\nСодержимое data:")
for item in DATA_DIR.iterdir():
    print(item)

BASE_DIR: /content/drive/MyDrive/smart-parking-vision
DATA_DIR: /content/drive/MyDrive/smart-parking-vision/data

Содержимое data:
/content/drive/MyDrive/smart-parking-vision/data/README.dataset.txt
/content/drive/MyDrive/smart-parking-vision/data/README.roboflow.txt
/content/drive/MyDrive/smart-parking-vision/data/test
/content/drive/MyDrive/smart-parking-vision/data/valid
/content/drive/MyDrive/smart-parking-vision/data/train


In [7]:
from pathlib import Path
import shutil

COLAB_DIR = Path("/content/smart-parking-vision")

if COLAB_DIR.exists():
    shutil.rmtree(COLAB_DIR)

COLAB_DIR.mkdir(exist_ok=True)

print(COLAB_DIR)

/content/smart-parking-vision


In [8]:
from pathlib import Path
import shutil
import time

DRIVE_BASE = Path("/content/drive/MyDrive/smart-parking-vision")
COLAB_BASE = Path("/content/smart-parking-vision")

start = time.time()

shutil.copytree(
    DRIVE_BASE / "data",
    COLAB_BASE / "data"
)

print("Копирование завершено")
print("Время:", round(time.time() - start, 2), "сек")

Копирование завершено
Время: 4527.53 сек


## Проверяем классы в COCO

In [9]:
import json
from pathlib import Path
from collections import Counter

BASE_DIR = Path("/content/smart-parking-vision")
DATA_DIR = BASE_DIR / "data"

for split in ["train", "valid", "test"]:
    ann_path = DATA_DIR / split / "_annotations.coco.json"

    print("\n")
    print("SPLIT:", split)

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    print("Категории:")
    for cat in coco["categories"]:
        print(cat)

    category_id_to_name = {
        cat["id"]: cat["name"] for cat in coco["categories"]
    }

    counter = Counter()

    for ann in coco["annotations"]:
        counter[category_id_to_name[ann["category_id"]]] += 1

    print("Количество объектов:")
    print(counter)



SPLIT: train
Категории:
{'id': 0, 'name': 'spaces', 'supercategory': 'none'}
{'id': 1, 'name': 'space-empty', 'supercategory': 'spaces'}
{'id': 2, 'name': 'space-occupied', 'supercategory': 'spaces'}
Количество объектов:
Counter({'space-empty': 265908, 'space-occupied': 231948})


SPLIT: valid
Категории:
{'id': 0, 'name': 'spaces', 'supercategory': 'none'}
{'id': 1, 'name': 'space-empty', 'supercategory': 'spaces'}
{'id': 2, 'name': 'space-occupied', 'supercategory': 'spaces'}
Количество объектов:
Counter({'space-empty': 73629, 'space-occupied': 69687})


SPLIT: test
Категории:
{'id': 0, 'name': 'spaces', 'supercategory': 'none'}
{'id': 1, 'name': 'space-empty', 'supercategory': 'spaces'}
{'id': 2, 'name': 'space-occupied', 'supercategory': 'spaces'}
Количество объектов:
Counter({'space-empty': 36584, 'space-occupied': 34100})


## Создаем чистую папку data_yolo

In [10]:
import shutil
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")
YOLO_DIR = BASE_DIR / "data_yolo"

if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

for split in ["train", "valid", "test"]:
    (YOLO_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print("Создана чистая папка:", YOLO_DIR)

Создана чистая папка: /content/smart-parking-vision/data_yolo


## Конвертация COCO в YOLO

In [11]:
import json
import shutil
from pathlib import Path
from collections import Counter

BASE_DIR = Path("/content/smart-parking-vision")

SOURCE_DATA_DIR = BASE_DIR / "data"
OUTPUT_DATA_DIR = BASE_DIR / "data_yolo"

SPLITS = ["train", "valid", "test"]

CLASS_NAME_TO_ID = {
    "space-empty": 0,
    "space-occupied": 1
}


def coco_bbox_to_yolo(bbox, img_width, img_height):
    x_min, y_min, box_w, box_h = bbox

    x_center = x_min + box_w / 2
    y_center = y_min + box_h / 2

    x_center /= img_width
    y_center /= img_height
    box_w /= img_width
    box_h /= img_height

    return x_center, y_center, box_w, box_h


def convert_split(split):
    source_split_dir = SOURCE_DATA_DIR / split
    output_images_dir = OUTPUT_DATA_DIR / split / "images"
    output_labels_dir = OUTPUT_DATA_DIR / split / "labels"

    annotation_path = source_split_dir / "_annotations.coco.json"

    print("\nSPLIT:", split)

    with open(annotation_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    category_id_to_name = {
        cat["id"]: cat["name"].lower().strip()
        for cat in coco["categories"]
    }

    image_id_to_info = {
        img["id"]: img
        for img in coco["images"]
    }

    image_id_to_annotations = {}

    for ann in coco["annotations"]:
        image_id = ann["image_id"]

        if image_id not in image_id_to_annotations:
            image_id_to_annotations[image_id] = []

        image_id_to_annotations[image_id].append(ann)

    copied_images = 0
    missing_images = 0
    object_counter = Counter()

    for image_id, img_info in image_id_to_info.items():
        file_name = img_info["file_name"]
        image_width = img_info["width"]
        image_height = img_info["height"]

        source_image_path = source_split_dir / file_name

        if not source_image_path.exists():
            missing_images += 1
            if missing_images <= 5:
                print("Не найдено изображение:", source_image_path)
            continue

        target_image_path = output_images_dir / Path(file_name).name

        if not target_image_path.exists():
            shutil.copy2(source_image_path, target_image_path)

        label_path = output_labels_dir / f"{Path(file_name).stem}.txt"

        yolo_lines = []

        for ann in image_id_to_annotations.get(image_id, []):
            category_name = category_id_to_name.get(
                ann["category_id"],
                "unknown"
            )

            if category_name not in CLASS_NAME_TO_ID:
                continue

            class_id = CLASS_NAME_TO_ID[category_name]

            x_center, y_center, box_w, box_h = coco_bbox_to_yolo(
                ann["bbox"],
                image_width,
                image_height
            )

            x_center = max(0, min(1, x_center))
            y_center = max(0, min(1, y_center))
            box_w = max(0, min(1, box_w))
            box_h = max(0, min(1, box_h))

            if box_w <= 0 or box_h <= 0:
                continue

            yolo_lines.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}"
            )

            object_counter[category_name] += 1

        with open(label_path, "w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))

        copied_images += 1

    print("Скопировано изображений:", copied_images)
    print("Не найдено изображений:", missing_images)
    print("Сконвертировано объектов:", object_counter)


for split in SPLITS:
    convert_split(split)

print("\nГотово. YOLO-датасет создан:", OUTPUT_DATA_DIR)


SPLIT: train
Скопировано изображений: 8691
Не найдено изображений: 0
Сконвертировано объектов: Counter({'space-empty': 265908, 'space-occupied': 231948})

SPLIT: valid
Скопировано изображений: 2483
Не найдено изображений: 0
Сконвертировано объектов: Counter({'space-empty': 73629, 'space-occupied': 69687})

SPLIT: test
Скопировано изображений: 1242
Не найдено изображений: 0
Сконвертировано объектов: Counter({'space-empty': 36584, 'space-occupied': 34100})

Готово. YOLO-датасет создан: /content/smart-parking-vision/data_yolo


## Проверка YOLO-датасета

In [12]:
from pathlib import Path
from collections import Counter

BASE_DIR = Path("/content/smart-parking-vision")
YOLO_DIR = BASE_DIR / "data_yolo"

for split in ["train", "valid", "test"]:
    images_dir = YOLO_DIR / split / "images"
    labels_dir = YOLO_DIR / split / "labels"

    image_files = list(images_dir.glob("*.jpg"))
    label_files = list(labels_dir.glob("*.txt"))

    class_counter = Counter()

    for label_path in label_files:
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    class_id = line.split()[0]
                    class_counter[class_id] += 1

    print("\nSPLIT:", split)
    print("images:", len(image_files))
    print("labels:", len(label_files))
    print("classes:", class_counter)


SPLIT: train
images: 8691
labels: 8691
classes: Counter({'0': 265908, '1': 231948})

SPLIT: valid
images: 2483
labels: 2483
classes: Counter({'0': 73629, '1': 69687})

SPLIT: test
images: 1242
labels: 1242
classes: Counter({'0': 36584, '1': 34100})


## Создаем YAML

In [13]:
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")

CONFIG_DIR = BASE_DIR / "configs"
CONFIG_DIR.mkdir(exist_ok=True)

yaml_text = """
path: /content/smart-parking-vision/data_yolo

train: train/images
val: valid/images
test: test/images

names:
  0: space-empty
  1: space-occupied
"""

yaml_path = CONFIG_DIR / "pklot_yolo.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_text.strip())

print("YAML создан:", yaml_path)
print(yaml_path.read_text())

YAML создан: /content/smart-parking-vision/configs/pklot_yolo.yaml
path: /content/smart-parking-vision/data_yolo

train: train/images
val: valid/images
test: test/images

names:
  0: space-empty
  1: space-occupied


## Проверка YAML

In [14]:
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")
yaml_path = BASE_DIR / "configs" / "pklot_yolo.yaml"

print("YAML существует:", yaml_path.exists())
print(yaml_path.read_text())

YAML существует: True
path: /content/smart-parking-vision/data_yolo

train: train/images
val: valid/images
test: test/images

names:
  0: space-empty
  1: space-occupied


## Проверка на 1 эпоху

In [15]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")
YAML_PATH = BASE_DIR / "configs" / "pklot_yolo.yaml"

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=1,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    cache=False,
    project=str(BASE_DIR / "runs"),
    name="parking_test_1_epoch",
    exist_ok=True,
    plots=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/smart-parking-vision/configs/pklot_yolo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s

## Итоговое обучение

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")
YAML_PATH = BASE_DIR / "configs" / "pklot_yolo.yaml"

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=40,
    imgsz=1024,
    batch=8,
    device=0,
    workers=2,
    cache=False,
    project=str(BASE_DIR / "runs"),
    name="parking_yolov8n_final",
    exist_ok=True,
    plots=True
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/smart-parking-vision/configs/pklot_yolo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parking_yolov8n_final, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

## Проверка обучения

In [17]:
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")
RUN_DIR = BASE_DIR / "runs" / "parking_yolov8n_final"

print("RUN_DIR:", RUN_DIR)
print("Существует:", RUN_DIR.exists())

for item in RUN_DIR.iterdir():
    print(item)

RUN_DIR: /content/smart-parking-vision/runs/parking_yolov8n_final
Существует: True
/content/smart-parking-vision/runs/parking_yolov8n_final/train_batch5441.jpg
/content/smart-parking-vision/runs/parking_yolov8n_final/labels.jpg
/content/smart-parking-vision/runs/parking_yolov8n_final/confusion_matrix.png
/content/smart-parking-vision/runs/parking_yolov8n_final/train_batch1.jpg
/content/smart-parking-vision/runs/parking_yolov8n_final/train_batch2.jpg
/content/smart-parking-vision/runs/parking_yolov8n_final/confusion_matrix_normalized.png
/content/smart-parking-vision/runs/parking_yolov8n_final/BoxP_curve.png
/content/smart-parking-vision/runs/parking_yolov8n_final/val_batch2_pred.jpg
/content/smart-parking-vision/runs/parking_yolov8n_final/BoxPR_curve.png
/content/smart-parking-vision/runs/parking_yolov8n_final/BoxF1_curve.png
/content/smart-parking-vision/runs/parking_yolov8n_final/val_batch0_pred.jpg
/content/smart-parking-vision/runs/parking_yolov8n_final/weights
/content/smart-parki

## Валидация модели

In [18]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")

MODEL_PATH = BASE_DIR / "runs" / "parking_yolov8n_final" / "weights" / "best.pt"
YAML_PATH = BASE_DIR / "configs" / "pklot_yolo.yaml"

model = YOLO(str(MODEL_PATH))

metrics = model.val(
    data=str(YAML_PATH),
    imgsz=640,
    batch=8,
    device=0,
    plots=True
)

print(metrics)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1623.4±659.5 MB/s, size: 69.7 KB)
val: Scanning /content/smart-parking-vision/data_yolo/valid/labels.cache... 2483 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2483/2483 801.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 311/311 9.0it/s 34.6s
                   all       2483     143316      0.998      0.998      0.995      0.963
           space-empty       2062      73629      0.998      0.997      0.995      0.971
        space-occupied       1967      69687      0.998      0.999      0.994      0.954
Speed: 1.1ms preprocess, 4.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_i

## Предсказания на тестовых картинках

In [19]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")

MODEL_PATH = BASE_DIR / "runs" / "parking_yolov8n_final" / "weights" / "best.pt"
TEST_IMAGES = BASE_DIR / "data_yolo" / "test" / "images"

model = YOLO(str(MODEL_PATH))

results = model.predict(
    source=str(TEST_IMAGES),
    conf=0.25,
    save=True,
    project=str(BASE_DIR / "runs"),
    name="parking_test_predictions",
    exist_ok=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1242 /content/smart-parking-vision/data_yolo/test/images/2012-09-11_15_53_00_jpg.rf.8282544a640a23df05bd245a9210e663.jpg: 640x640 29 space-emptys, 72 space-occupieds, 7.2ms
image 2/1242 /content/smart-parking-vision/data_yolo/test/images/2012-09-11_16_48_36_jpg.rf.4ecc8c87c61680ccc73edc218a2c8d7d.jpg: 640x640 25 space-emptys, 77 space-occupieds, 7.1ms
image 3/1242 /content/smart-parking-vision/data_yolo/test/images/2012-09-12_06_36_36_jpg.rf.0886

### Копирование модели для проекта

In [20]:
import shutil
from pathlib import Path

BASE_DIR = Path("/content/smart-parking-vision")

source_model = BASE_DIR / "runs" / "parking_yolov8n_final" / "weights" / "best.pt"

models_dir = BASE_DIR / "models"
models_dir.mkdir(exist_ok=True)

target_model = models_dir / "parking_yolov8n_best.pt"

shutil.copy2(source_model, target_model)

print("Модель скопирована:")
print(target_model)

Модель скопирована:
/content/smart-parking-vision/models/parking_yolov8n_best.pt
